In [1]:
import io
import urllib.request
import zipfile
from pathlib import Path

import pandas as pd

csv_path = Path("data") / "freMTPL2freq.csv"
if not csv_path.exists():
    url = "https://github.com/ds-careers-ominimo/ominimo-careers-data/raw/main/ml_claims_forecasting.zip"
    archive = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url).read()))
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    csv_path.write_bytes(archive.read("freMTPL2freq.csv"))

claims = pd.read_csv(csv_path)
print(claims.shape)

(678013, 12)


In [2]:
feature_columns = ["Area", "VehPower", "VehAge", "DrivAge", "BonusMalus", "VehBrand", "VehGas", "Density", "Region"]

print(claims.isna().sum())
print(f"IDpol duplicated {claims['IDpol'].duplicated().sum()} unique {claims['IDpol'].nunique()} rows {len(claims)}")
print(claims["ClaimNb"].value_counts().sort_index())
print(f"ClaimNb zero share {(claims['ClaimNb'] == 0).mean():.4f} max {claims['ClaimNb'].max()} above 4 {(claims['ClaimNb'] > 4).sum()}")
print(claims["Exposure"].agg(["min", "median", "mean", "max"]))
print(f"Exposure above 1 {(claims['Exposure'] > 1).sum()} below 0.02 {(claims['Exposure'] < 0.02).sum()}")
duplicated_features = claims.duplicated(subset=feature_columns, keep=False).sum()
duplicated_with_exposure = claims.duplicated(subset=feature_columns + ["Exposure"], keep=False).sum()
print(f"duplicate rows on 9 feature columns {duplicated_features} share {duplicated_features / len(claims):.3f}")
print(f"duplicate rows on 9 feature columns plus Exposure {duplicated_with_exposure} share {duplicated_with_exposure / len(claims):.3f}")
print(claims[["VehPower", "VehAge", "DrivAge", "BonusMalus", "Density"]].agg(["min", "max"]))
print(f"VehAge above 30 {(claims['VehAge'] > 30).sum()} DrivAge above 90 {(claims['DrivAge'] > 90).sum()} BonusMalus above 150 {(claims['BonusMalus'] > 150).sum()}")
implied_frequency = claims["ClaimNb"] / claims["Exposure"]
print(f"implied frequency max {implied_frequency.max():.1f} above 20 {(implied_frequency > 20).sum()}")
print(f"portfolio frequency {claims['ClaimNb'].sum() / claims['Exposure'].sum():.6f}")

IDpol         0
ClaimNb       0
Exposure      0
Area          0
VehPower      0
VehAge        0
DrivAge       0
BonusMalus    0
VehBrand      0
VehGas        0
Density       0
Region        0
dtype: int64
IDpol duplicated 0 unique 678013 rows 678013
ClaimNb
0     643953
1      32178
2       1784
3         82
4          7
5          2
6          1
8          1
9          1
11         3
16         1
Name: count, dtype: int64
ClaimNb zero share 0.9498 max 16 above 4 9
min       0.002732
median    0.490000
mean      0.528750
max       2.010000
Name: Exposure, dtype: float64
Exposure above 1 1224 below 0.02 13603


duplicate rows on 9 feature columns 257911 share 0.380
duplicate rows on 9 feature columns plus Exposure 41505 share 0.061
     VehPower  VehAge  DrivAge  BonusMalus  Density
min         4       0       18          50        1
max        15     100      100         230    27000
VehAge above 30 1116 DrivAge above 90 401 BonusMalus above 150 209
implied frequency max 732.0 above 20 1214
portfolio frequency 0.100703
